# Testing, Formatting, Linting, Type-Checking, and Continuous Integration

Adapted from material by
[Eva Maxfield Brown](https://github.com/evamaxfield)

## Structure

Talk a tiny bit about each topic, but the main focus is showing how to add each of these processes to your code and how they work.

In between each section I will let everyone try to add the step we just discussed to their own code. If you run into issues or are confused, let us know. We can then answer that question for the whole group.

## Motivation

This is 'expected knowledge' for software developers, but it's rarely taught to scientists. Yet it's exactly what keeps scientific code trustworthy: tests catch when a change silently breaks a result, and CI runs those checks automatically on every change.

Understanding testing and continuous integration (CI) well is also a highly transferable skill — it shows up in nearly every software engineering role.

---

Regardless of industry or academia, CI systems can run lots of different tasks — whole processing pipelines, docs builds, releases — and they're free for open-source projects like Cantera.

_(Please use CI responsibly — these are shared, free resources.)_

## Testing

### Testing

Testing consist in writing small functions that test:

* whether small units of your code function as expected
* whether these small units integrate well together
* whether your code takes care of edge cases
* whether your code's inputs and outputs are correctly treated

Running your tests allow checking that new changes to the code base did not break anything in your code (at least what your are testing for!).

### Testing

**As you write tests, you get to experience what it takes to use your library and this might lead you to refactor parts of your code; refactoring is an important part of a software life-cycle!**

### Testing — types of tests

Tests are usually grouped by **scope** — how much of the system one test exercises:

* **Unit** — a single function or class, in isolation
* **Integration** — several units working together
* **Functional / end-to-end** — a whole workflow, from a user's point of view

You don't need all of these on day one. For most scientific code, **start with unit tests** — they catch the most bugs for the least effort.

### Testing — unit tests

The smallest scope: test **one function or class on its own**.

* Fast, numerous, and pinpoint *exactly* what broke
* No external resources (files, network, databases)
* The bulk of most test suites

Our [`test_rescale`](../tests/test_rescale.py) is a unit test — it calls `rescale()` with a known array and asserts the exact output:

```python
def test_rescale() -> None:
    input_array = np.array([1, 2, 3, 4, 5])
    output_array = rescale(input_array)
    expected_array = np.array([0, 0.25, 0.5, 0.75, 1])
    np.testing.assert_allclose(output_array, expected_array)
```

### Testing — integration & functional tests

Bigger scope, fewer of them:

* **Integration** — check that units work *together*. e.g. creating a multi-reactor network: do the pieces connect correctly?
* **Functional / end-to-end (E2E)** — check a *whole workflow* from the user's point of view: given this input file, does the full pipeline produce the expected output?

These catch bugs unit tests can't — wrong assumptions about how components fit together — but they're slower and harder to debug.

This is the **"test pyramid"**: many fast unit tests at the base, fewer integration tests above, and only a handful of slow end-to-end tests at the top.

### Testing — regression tests

Grouped by **purpose**, not scope: a regression test makes sure a **bug that was fixed stays fixed**.

The workflow when you find a bug:

1. Write a test that **reproduces** it — it fails, proving the bug is real
2. Fix the code until the test passes
3. **Keep the test** — it now guards against the bug ever coming back

e.g. `rescale` divides by zero when every input value is equal (`H == L`). A regression test would feed it a constant array and assert it behaves sensibly — so a future change can't silently reintroduce that crash.

### Testing — property-based testing

Instead of hand-picking inputs, state a **property that must always hold** and let a tool generate many inputs trying to break it.

This shines for scientific code, where outputs obey mathematical rules:

* `rescale`'s output should *always* lie in `[0, 1]`
* a sort should *always* return a list of the same length

### Testing

How to write tests — let's look at the real files in this repo:

* [`tests/test_rescale.py`](../tests/test_rescale.py) — a simple test, a `@pytest.mark.parametrize` test, and a fixture-based test
* [`tests/conftest.py`](../tests/conftest.py) — the `preloaded_data` fixture

Run them:

```bash
pytest tests/       # or: just test
```

## Formatting

### Formatting

Formatting does two things:

1. Stops all the arguments about: tabs vs spaces, new lines in the middle of operations, etc. NO MORE STUPID ARGUMENTS!

MORE IMPORTANTLY

2. Keeps a consistent style across the whole project. Whether you wrote the code or someone else, it will look the same.

### Formatting

How to format your code — it's already wired into this repo:

* [`.pre-commit-config.yaml`](../.pre-commit-config.yaml) — the **black** hook
* [`pyproject.toml`](../pyproject.toml) — `[tool.black]` (line length, target versions)

Run it:

```bash
black --check --diff messy_code.py
```

## Linting

### Linting

Formatting focuses on directly changing the code to some "standard" style. Linting looks for common problems in the code.

A nice example of the difference is that formatting won't change the following:

```python
from package_a import foo
from package_b import bar
from package_a import foo, baz
```

But linting should inform you (if not automatically fix) that `foo` from `package_a` is imported twice and a nicer import block should look like:

```python
from package_a import baz, foo  # alphabetical
from package_b import bar
```

### Linting

* It will also alert you of unused variables that you may have forgot to use or accidentially left from debugging etc.<br><br>
* It will remind you about function and module documentation standards.<br><br>
* It will help you fix code which is doing extra bits of work (i.e. accidental double looping / possible standard library replacements).

I won't list all of the rules it checks against but there are [A LOT](https://docs.astral.sh/ruff/rules/).

### Linting

How to lint your code — already configured here:

* [`.pre-commit-config.yaml`](../.pre-commit-config.yaml) — the **ruff** hook (runs with `--fix`)
* [`pyproject.toml`](../pyproject.toml) — `[tool.ruff]` (the rule sets we enable and ignore)

Run it:

```bash
ruff check --diff messy_code.py
```

## Type Checking

### Type Checking

Type checking is the most thorough analysis of your code you can do before you even run it.

Python added **optional** types in version 3.5. At first they were simply for analysis of the code prior to running to check for even more bugs and possible error cases but in the latest versions of Python, they are now being used to speed up your programs.

They can be annoying to add and work with. Very annoying, because they are optional in the language so there is a lot of hacky ways they can be manipulated.

### Type Checking

Typing is handled by decorations in the code i.e.

In [7]:
def example(a):
    print(f"Hello {a}")

def example_typed(a: str) -> None:
    print(f"Hello {a}")

### Type Checking

And they can get complicated...

In [8]:
import random

# yay only a single value
var_int: int = 5
var_float: float = 0.5
var_bool: bool = False
var_string: str = "wow!"

# oh no multiple values
list_of_strings: list[str] = ["hello", "world"]
list_of_mixed: list[str | int | bool] = ["hello", 3, True]

# lists don't have any constraints on size but tuples do
tuple_of_two_values: tuple[str, int] = ("number", 10)
tuple_of_n_values: tuple[str, ...] = tuple(["python why did you add this" for i in range(random.randint(1, 5))])
print(tuple_of_n_values)

# sub-objects
dict_of_dicts: dict[str, dict[str, str | int]] = {
    "eva": {"name": "eva maxfield brown", "age": 28},
    "bob": {"name": "Bob Boberson"},
}

('python why did you add this', 'python why did you add this', 'python why did you add this')


### Type Checking

An example with our favourite pattern, the factory:

In [9]:
from abc import ABC, abstractmethod
from typing import Type

class Localizer(ABC):

    @abstractmethod
    def localize(self, text: str) -> str:
        raise NotImplementedError()

class EnglishLocalizer(Localizer):

    def localize(self, text: str) -> str:
        return "Hello world"

localizers = {
    "en": EnglishLocalizer,
}

def get_localizer_not_initialized(lang: str) -> Type[Localizer]:
    return localizers[lang]

def get_localizer_initialized(lang: str) -> Localizer:
    return localizers[lang]()

### Type Checking

Some tips / don't worry too much...

**You rarely have to hand-write hard types — libraries ship their own.**

`numpy`, `pandas`, and most scientific libraries provide types you can reuse directly in your annotations.

In [10]:
import numpy as np

# numpy, pandas, and most scientific libraries ship their own types --
# you just reuse them in your annotations, no need to hand-write anything:
var_array: np.ndarray = np.random.random((3, 2))
var_array

array([[0.52455782, 0.92397195],
       [0.39290097, 0.83134081],
       [0.28173029, 0.46499526]])

### Type Checking 

**When nesting gets ugly, reach for a `dataclass`.**

A deeply nested `dict[str, dict[str, ...]]` is technically correct but hard to read, and the checker can't tell you which keys must exist. Define the sub-object once — with named, typed fields — and it's clearer *and* safer.

In [11]:
from dataclasses import dataclass

# Technically correct, but hard to read -- and the checker can't tell you
# which keys must exist or catch a misspelled key:
dict_of_dicts: dict[str, dict[str, str | int]] = {
    "alice": {"name": "Alice Smith", "age": 30},
    "bob": {"name": "Bob Boberson"},
}

# Define the sub-object once, with named + typed fields:
@dataclass
class PersonDetails:
    name: str
    age: int | None = None

people: dict[str, PersonDetails] = {
    "alice": PersonDetails(name="Alice Smith", age=30),
    "bob": PersonDetails(name="Bob Boberson"),
}

# The payoff: attribute access is typed and checked
# (autocompletes, and a typo like .agee would be flagged)
people["alice"].age

30

### Type Checking

It is hard to learn but do practice using it. It will make writing code easier the more you do it as it will check a lot of assumptions for you.

Some resources to continue learning / lookup in the future:

* [standard library docs](https://docs.python.org/3.11/library/typing.html)
* [typing cheatsheet from mypy](https://mypy.readthedocs.io/en/stable/cheat_sheet_py3.html)

### Type Checking

How to type-check your code — already configured here:

* [`.pre-commit-config.yaml`](../.pre-commit-config.yaml) — the **mypy** hook
* [`pyproject.toml`](../pyproject.toml) — `[tool.mypy]` (`disallow_untyped_defs`, `check_untyped_defs`, ...)
* [`src/py.typed`](../src/py.typed) — marks the package as typed so downstream users get type info

Run it:

```bash
mypy messy_code.py
```

## Continuous Integration

### Continuous Integration

Continuous Integration (CI) is meant to reduce or remove bugs entering code over time. Whether the code base is changing (new features, bug fixes, etc.) or upstream dependencies are changing (new releases).

The main idea is that by checking your tests, formatting, linting, and types for each commit / PR you we know that nothing is breaking.

If something does break, you know the exact commit / PR which broke something.

### Continuous Integration

Further, it allows you to test on more machine setups than your own. For example, I have a Linux laptop and a Windows desktop and I use Python 3.11 on both. But I have users who use MacOS on Intel chip and Python 3.9.

CI systems (GitHub Actions, GitLab CI/CD, Azure Pipelines, etc.) allow you to run the same suite of tests across all of these machines setups.

Fortunately, once you have testing, formatting, linting, and type-checking setup locally, it is pretty easy to add CI to your repo!

### Continuous Integration

How CI is set up in this repo:

* [`.github/workflows/ci.yml`](../.github/workflows/ci.yml) — a **test** job (matrix across Linux/macOS/Windows and Python 3.10–3.12) and a **lint** job (runs `pre-commit` on all files)

It runs automatically on every push and pull request to `main`.

### Continuous Integration — separate workflows by *when* they run

Don't cram everything into one workflow. Split them by **trigger and purpose**. This repo has two:

* [`ci.yml`](../.github/workflows/ci.yml) — tests + lint. Runs on **every push and pull request**, so you get feedback on *every* change.
* [`publish-slides.yml`](../.github/workflows/publish-slides.yml) — builds & deploys the slides. Runs **only on push to `main`, and only when `notebooks/**` changes** (a `paths:` filter) — no point rebuilding slides for a code-only change.

**Why separate them?**

* **Right trigger for the job** — checks belong on PRs; deploys belong on `main` only (you don't want every PR publishing).
* **Save time & resources** — `paths:` filters skip work the change can't affect.
* **Independent failures** — a broken slide build doesn't block a code PR, and vice versa.
* **Least privilege** — deploying needs `pages: write` / `id-token: write`; testing doesn't. Keeping them apart limits what each job can do.

## Code Coverage


### Code Coverage

Once you have tests, a natural question is: **how much of my code do they actually exercise?**

*Code coverage* measures which lines (and branches) of your source code are run when your test suite executes. It is usually reported as a percentage:

* **Line coverage** — what fraction of lines were executed
* **Branch coverage** — what fraction of `if`/`else` branches were taken

A line that is never run by any test is a line whose behavior you are not verifying.


### Code Coverage

A word of caution: **high coverage is not the same as good tests.**

* 100% coverage only means every line *ran* — not that you asserted the right things
* It is easy to "cover" a line without meaningfully testing it
* Chasing a coverage number can lead to lots of low-value tests

Use coverage as a **guide** to find untested code, not as a goal in itself. A realistic target plus thoughtful tests beats 100% of trivial ones.


### Code Coverage

We measure coverage with [`pytest-cov`](https://pytest-cov.readthedocs.io/), a plugin built on [`coverage.py`](https://coverage.readthedocs.io/).

Add it to the `test` dependencies in `pyproject.toml`:

```toml
[project.optional-dependencies]
test = [
    "pytest>=7,<8",
    "pytest-cov",
]
```

Then run your tests with coverage reporting:

```bash
pytest tests/ --cov=rescale --cov-report=term-missing
```

`--cov=rescale` measures the `rescale` package and `--cov-report=term-missing` prints which lines were *not* covered.


### Code Coverage

You can configure `coverage.py` in `pyproject.toml` so everyone gets the same settings


### Code Coverage

Finally, add coverage to CI so every commit / PR reports it.

A coverage job generates a machine-readable report and uploads it to a service such as [Codecov](https://about.codecov.io/) (via [`codecov/codecov-action`](https://github.com/codecov/codecov-action)), which then comments on each PR showing how coverage changed:

```yaml
  coverage:
    runs-on: ubuntu-latest
    steps:
    - uses: actions/checkout@v4
    - uses: actions/setup-python@v5
      with:
        python-version: "3.11"
    - run: pip install .[test]
    - run: pytest tests/ --cov=rescale --cov-report=xml
    - uses: codecov/codecov-action@v4
```

Uploading to Codecov needs a `CODECOV_TOKEN` repository secret, which a repo **admin** configures from the Codecov dashboard.


## Packaging & Distribution


### Packaging & Distribution

We've already seen how to *build* a package — here we focus on how it fits into **CI/CD**.

The idea: releasing should be as automated and repeatable as testing. A release job lives alongside the `test`, `lint`, and `coverage` jobs, but it runs only when you cut a release — typically triggered by a **version tag** (e.g. `v0.1.0`) or a published GitHub Release, not on every commit.

Notice our CI workflow already listens for tags:

```yaml
on:
  push:
    tags:
      - "v*"
```


### Packaging & Distribution

When triggered, the release job builds the distribution and **publishes it to [PyPI](https://pypi.org/)** so anyone can `pip install rescale`.

* It uploads via [`pypa/gh-action-pypi-publish`](https://github.com/pypa/gh-action-pypi-publish).
* Authentication uses **PyPI Trusted Publishing** (OIDC) — PyPI trusts our GitHub repo directly, so no API-token secret is needed.

> *We won't actually publish `rescale` to PyPI in this course — this is conceptual. But the tag trigger and the release-job pattern are exactly how a real project ships to PyPI.*


### Packaging & Distribution

See the conceptual `release` job in `.github/workflows/ci.yml`. It's guarded with `if: false` so it **never runs** — flip the guard off (and set up Trusted Publishing) when you're ready to ship.


## Everything At Once

### Everything At Once

Everything we just discussed lives together in this repo. The [`Justfile`](../Justfile) ties it together:

```bash
just install   # install the package with lint, test, and dev extras
just test      # run the test suite
just lint      # run formatting, linting, and type-checking via pre-commit
```

and CI ([`.github/workflows/ci.yml`](../.github/workflows/ci.yml)) runs the same checks on every push and PR.

## Credit

Much of the written content came directly from [https://pydev-guide.github.io/](https://pydev-guide.github.io/).

It is still under development but some tutorials are already available. I highly recommend starring it / checking back to it every few months.

These slides are adapted from material created by [Eva Maxfield Brown](https://github.com/evamaxfield) for the URSSI Winter School ([winter-school-lectures](https://github.com/evamaxfield/winter-school-lectures)).